# Lab 6 · RAG: ground the answers in the policy

**Time:** 30 min · **You will:** see Retrieval-Augmented Generation built from scratch — split the policy, embed it, retrieve the relevant sections for each question, and make the SLM answer only from them. Then measure how much it helps.

**Idea:** the model doesn't need to *know* ABC's policy if we *show* it the right paragraph at question time.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**, read each result, then change the `🔧` values and run those cells again (Shift+Enter).*

In [ ]:
import sys, os
# find the course folder (the one holding slm_lab.py), whichever folder this notebook was opened from
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import *
import pandas as pd
from sentence_transformers import SentenceTransformer

sections = policy_sections()
print(len(sections), "sections")
for s in sections[:4]:
    print(f"- {s['title']}: {s['text'][:80]}…")

## 6.1 Embeddings: meaning as vectors
An **embedding model** maps a text to a vector so that texts with similar meaning are close together. We compare vectors with **cosine similarity** (1 = same direction, 0 = unrelated).

In [ ]:
embedder = SentenceTransformer(EMBED_ID, device=device())
section_texts = [f"{s['title']}. {s['text']}" for s in sections]
section_vecs = embedder.encode(section_texts, normalize_embeddings=True, convert_to_tensor=True)
print("section vectors:", tuple(section_vecs.shape))

### 🔧 Try it 6.1 — The retriever
`retrieve(question, k)` returns the `k` most similar policy sections with their similarity scores, best first. Because the vectors are normalised, cosine similarity is just a dot product: `section_vecs @ q_vec`.

🔧 **Try:** change `my_question` to *"Can I work from home?"* or *"Is there a free gym?"* (not in the policy — look at the scores).

In [ ]:
def retrieve(question, k=3):
    q_vec = embedder.encode(question, normalize_embeddings=True, convert_to_tensor=True)
    scores = section_vecs @ q_vec
    top = torch.topk(scores, k)
    return [(float(s), sections[int(i)]) for s, i in zip(top.values, top.indices)]

my_question = "I'm becoming a father - how much leave can I take?"   # 🔧 ask anything
for score, s in retrieve(my_question):
    print(f"{score:.2f}  {s['title']}")

In [ ]:
hits = retrieve("I'm becoming a father - how much leave can I take?")
check(hits[0][1]["title"].startswith("4.3") and hits[0][0] >= hits[1][0],
      "Found Paternity Leave without the word 'paternity' in the question - that's semantic search.",
      "the paternity section was not the top result.")

### 👀 See it 6.2 — Measure retrieval on its own (recall@k)
If retrieval misses the right section, the model can't answer — so we measure it separately. For every **fact** question in the test set, the cell checks whether its correct section is among the top `k` results, for k = 1, 2 and 3.

In [ ]:
section_of = {f["id"]: f["section"] for f in load_facts()}
fact_items = [it for it in eval_items() if it["kind"] == "fact"]
recall = {}
for k in (1, 2, 3):
    found = [any(s["title"] == section_of[it["id"]] for _, s in retrieve(it["question"], k)) for it in fact_items]
    recall[k] = sum(found) / len(found)
recall

In [ ]:
check(recall[3] >= recall[1],
      f"Top-1 finds the right section {recall[1]:.0%} of the time, top-3 {recall[3]:.0%}. We'll pass 3 sections "
      "to the model so a near-miss still works.",
      "recall@3 should never be lower than recall@1.")

## 6.2 Generate from the retrieved context
RAG only works if the model **follows the excerpts and admits when they do not contain the answer**. The small 0.5B model of the earlier labs is weak at both, so from here on (Labs 6, 7, 13 and 21) we use the **3B model** of the same family (about 6 GB, downloaded once). We test it with and without the policy, so the comparison is fair.

In [ ]:
MODEL_3B = "Qwen/Qwen2.5-3B-Instruct"
tok, model = load(MODEL_3B)
UNKNOWN = UNKNOWN_REPLY

def rag_messages(question, k=3):
    context = "\n\n".join(f"[Section {s['title']}]\n{s['text']}" for _, s in retrieve(question, k))
    system = ("You are the HR assistant of ABC Company. Answer the question in one sentence using ONLY the "
              "policy excerpts below, and cite the section number.\n"
              f"If the excerpts do not contain the answer, reply exactly: {UNKNOWN}\n\n"
              f"Policy excerpts:\n{context}")
    return [{"role": "system", "content": system}, {"role": "user", "content": question}]

print(rag_messages("How many sick days do I get?")[0]["content"][:600])

### 🔧 Try it 6.3 — RAG answers + evaluation
`ask_rag(questions)` builds the RAG prompt for each question and answers with the model. First ask one question yourself, then the cell scores the whole test set and compares with the Lab 5 baseline.

🔧 **Try:** change `my_question` and compare with the plain model's answer from Lab 1.

In [ ]:
def ask_rag(questions):
    return chat(model, tok, [rag_messages(q) for q in questions], max_new_tokens=80)

my_question = "How many days of paternity leave do I get?"   # 🔧 ask anything
print(my_question, "→", ask_rag([my_question])[0])

In [ ]:
with Timer() as t:
    acc_rag, res_rag = run_eval(ask_rag)
# the same 3B model WITHOUT the policy, so the comparison is fair (Lab 5's baseline used the smaller 0.5B model)
acc_plain, res_plain = run_eval(lambda qs: chat(model, tok, qs, system=HR_SYSTEM, max_new_tokens=80))
base = acc_plain
print(f"3B model with the policy (RAG): {acc_rag:.0%}   ·   same model without the policy: {acc_plain:.0%}")
pd.DataFrame({"with policy (RAG)": res_rag.groupby("kind")["correct"].mean(), "without policy": res_plain.groupby("kind")["correct"].mean()})

In [ ]:
check(len(res_rag) == len(eval_items()) and acc_rag > acc_plain,
      f"Grounding the same model in the policy lifts accuracy from {acc_plain:.0%} to {acc_rag:.0%}. Look at the two kinds: questions the policy answers, and questions it does not (the right reply there is 'I don't know').",
      "RAG did not beat the same model without the policy - read the table above to see which kind of question went wrong.")
prompt_tokens = sum(len(tok.encode(tok.apply_chat_template(rag_messages(it["question"]), tokenize=False)))
                    for it in eval_items()) / len(eval_items())
save_report("rag", {"accuracy": round(float(acc_rag), 3),
                    "fact_accuracy": round(float(res_rag[res_rag.kind == "fact"].correct.mean()), 3),
                    "unknown_accuracy": round(float(res_rag[res_rag.kind == "unknown"].correct.mean()), 3),
                    "avg_prompt_tokens": round(prompt_tokens), "recall@3": round(recall[3], 3),
                    "seconds_per_question": round(t.seconds / len(eval_items()), 3)})

In [ ]:
pd.set_option("display.max_colwidth", 140)
res_rag[~res_rag.correct][["question", "answer"]]

### 🔧 Try it 6.4 — Refuse before asking the model
For unknown topics, the best section's similarity score is usually low. Below a **threshold** we can answer `UNKNOWN` directly, without calling the model. The cell prints the top-1 scores for fact vs unknown questions and picks the threshold just below the lowest fact score.

🔧 **Try:** replace the threshold with your own number, e.g. `threshold = 0.40`. A higher threshold catches more unknown questions — but starts blocking real ones.

In [ ]:
items = eval_items()
top1 = pd.DataFrame({"kind": [it["kind"] for it in items],
                     "score": [retrieve(it["question"], 1)[0][0] for it in items]})
print(top1.groupby("kind")["score"].describe()[["min", "mean", "max"]])

facts_min = top1[top1.kind == "fact"].score.min()
threshold = round(float(facts_min) - 0.01, 2)   # 🔧 or type your own number, e.g. 0.40

caught = (top1[top1.kind == "unknown"].score < threshold).mean()
lost = (top1[top1.kind == "fact"].score < threshold).mean()
print(f"threshold {threshold}: catches {caught:.0%} of unknown questions, wrongly blocks {lost:.0%} of fact questions")

In [ ]:
check(lost <= 0.1,
      f"A threshold of {threshold} blocks {caught:.0%} of off-policy questions for free. The rest still rely on "
      "the model's instruction-following - a cheap filter plus the model is a common production pattern.",
      f"a threshold of {threshold} blocks {lost:.0%} of real policy questions - that's too high.")

## 🧠 Check your understanding
1. Name two ways RAG can still give a wrong answer.
2. RAG made each prompt much longer. What does that cost, and when would it become a problem?
3. When would you prefer RAG over fine-tuning, even if fine-tuning scored higher?

*✍️ Your answer:* 

**Next:** Lab 7 — RAG experiments: change one thing and watch what breaks.